# História dos clusters — Bahia 2016 a 2025

**Unidade:** município × semana epidemiológica, replicada em **10 anos**.

**Pergunta:** a escada v0→v5 se comporta de forma parecida ao longo do tempo? Os perfis e a qualidade interna (silhouette) mudam com o volume de notificações?

Mesma metodologia do notebook `02_historia_clusters.ipynb`, estendida ano a ano.

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

_cwd = Path.cwd().resolve()
if (_cwd / "ml").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "ml").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError(
        "Não achei a pasta ml/. Abra o notebook a partir de ia-iv/ ou ia-iv/notebooks/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for name in list(sys.modules):
    if name == "ml" or name.startswith("ml."):
        del sys.modules[name]

from ml.cluster import run_kmeans
from ml.columns import Col, Feat, FEATURE_SETS
from ml.config import DEFAULT_K, DEFAULT_MIN_CASOS_ANUAL, DEFAULT_YEARS, REFERENCE_POP_YEAR
from ml.dataset import build_features_panel, load_region_raw
from ml.paths import region_manifest_path
from ml.preprocess import build_region_parquet
from ml.regions import BA
from ml.validate import transition_matrix

VERSION_BLOCKS = {
    "v0": "baseline: casos",
    "v1": "+ incidência por 100 mil",
    "v2": "+ média móvel (3 semanas)",
    "v3": "+ crescimento semanal",
    "v4": "+ aceleração",
    "v5": "+ densidade populacional (hab/km²)",
}

REGION = BA.slug
YEARS = list(range(2016, 2026))  # 2016 a 2025 (10 anos)
K = DEFAULT_K
VERSIONS = ["v0", "v1", "v2", "v3", "v4", "v5"]
VERSION_FINAL = "v5"
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00"][:K]
ACCENT = "#0072B2"
SCATTER_KW = dict(s=8, alpha=0.35, linewidths=0.2, edgecolors="#333333")

print(f"Projeto: {ROOT}")
print(f"Python: {sys.executable}")
print(f"Região: {BA.name} | Anos: {YEARS[0]} a {YEARS[-1]} | K={K}")
print(f"População/densidade: referência fixa IBGE {REFERENCE_POP_YEAR}")

## Ato 1 — Parquet agregado BA

Um único `dengue.parquet` (2016 a 2025). Features v0…v5 são calculadas **em memória**; só o parquet bruto e `populacao.parquet` ficam em disco.

In [ ]:
build_region_parquet(BA, DEFAULT_YEARS)
manifest = json.loads(region_manifest_path(REGION).read_text(encoding="utf-8"))

coverage_rows = []
for year in YEARS:
    raw = load_region_raw(REGION, year)
    casos_anual = raw.groupby(Col.ID_MUNICIP).size()
    n_eleg = int((casos_anual >= DEFAULT_MIN_CASOS_ANUAL).sum())
    coverage_rows.append({
        "ano": year,
        "notificações": manifest["registros_por_ano"].get(str(year), len(raw)),
        "municípios elegíveis": n_eleg,
        "semanas": raw[Col.SEM_NOT].nunique(),
    })

coverage = pd.DataFrame(coverage_rows)
print(f"Total agregado: {manifest['registros_total']:,} notificações")
display(coverage)

fig, ax1 = plt.subplots(figsize=(9, 3.5))
ax1.bar(coverage["ano"].astype(str), coverage["notificações"], color=ACCENT, alpha=0.85)
ax1.set_ylabel("Notificações")
ax1.set_title("Volume de notificações — BA")
plt.tight_layout()

## Ato 2 — Escada v0→v5 em cada ano

Roda K-means ($K=4$) para todas as combinações ano × versão. Features ficam cacheadas em disco.

In [ ]:
def run_year_version(year: int, version: str, *, keep_panel: bool = False) -> dict:
    panel = build_features_panel(REGION, year, version)
    result = run_kmeans(panel, version, k=K)
    out = {
        "ano": year,
        "versão": version,
        "bloco": VERSION_BLOCKS[version],
        "silhouette": result.metrics["silhouette"],
        "davies_bouldin": result.metrics["davies_bouldin"],
        "n_municipios": panel[Col.ID_MUNICIP].nunique(),
        "n_linhas": len(panel),
        "cluster_means": result.cluster_means,
    }
    if keep_panel:
        out["panel"] = panel
        out["labels"] = result.labels
    return out

records: list[dict] = []
for year in YEARS:
    print(f"Ano {year}…", flush=True)
    for version in VERSIONS:
        records.append(run_year_version(year, version))

metrics = pd.DataFrame([{
    "ano": r["ano"],
    "versão": r["versão"],
    "silhouette": r["silhouette"],
    "davies_bouldin": r["davies_bouldin"],
    "n_municipios": r["n_municipios"],
    "n_linhas": r["n_linhas"],
} for r in records])

metrics_by = {(r["ano"], r["versão"]): r for r in records}
display(metrics.pivot(index="ano", columns="versão", values="silhouette").round(4))

### Silhouette: heatmap e tendência v0 vs v5

In [ ]:
sil_pivot = metrics.pivot(index="ano", columns="versão", values="silhouette")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.heatmap(
    sil_pivot,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    ax=axes[0],
    cbar_kws={"label": "silhouette"},
)
axes[0].set_title("Silhouette por ano e versão")
axes[0].set_xlabel("versão")
axes[0].set_ylabel("ano")

axes[1].plot(sil_pivot.index, sil_pivot["v0"], "o-", label="v0", color="#0072B2")
axes[1].plot(sil_pivot.index, sil_pivot[VERSION_FINAL], "s-", label="v5", color="#D55E00")
axes[1].set_xlabel("ano")
axes[1].set_ylabel("silhouette")
axes[1].set_title("v0 vs v5 ao longo dos anos")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()

### Δ silhouette (v5 − v0) por ano

Quanto a escada multivariada “custa” em métrica interna em cada ano.

In [ ]:
delta = metrics.groupby("ano").apply(
    lambda g: g.loc[g["versão"] == "v5", "silhouette"].iloc[0]
    - g.loc[g["versão"] == "v0", "silhouette"].iloc[0]
).rename("Δ silhouette (v5−v0)")

display(delta.round(4).to_frame())

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(delta.index.astype(str), delta.values, color=ACCENT, alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Δ silhouette")
ax.set_title("Queda de silhouette ao ir de v0 para v5")
plt.tight_layout()

## Ato 3 — Perfis v5 em anos selecionados

Compara médias de cluster nos anos com mais notificações e um ano calmo.

In [ ]:
# anos com pico e referência recente (ajuste se quiser)
peak_years = (
    coverage.nlargest(3, "notificações")["ano"].tolist()
    if len(coverage) >= 3
    else YEARS[-3:]
)
compare_years = sorted(set([YEARS[0], *peak_years, YEARS[-1]]))

profile_frames = []
for year in compare_years:
    r = metrics_by[(year, VERSION_FINAL)]
    means = r["cluster_means"].round(2).copy()
    means["ano"] = year
    means["cluster"] = means.index
    profile_frames.append(means.reset_index(drop=True))

profiles = pd.concat(profile_frames, ignore_index=True)
display(
    profiles[
        ["ano", "cluster", Feat.INCIDENCIA_100K, Feat.CRESCIMENTO, Feat.DENSIDADE_KM2, Feat.CASOS]
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
for year in compare_years:
    sub = profiles[profiles["ano"] == year]
    ax.scatter(
        sub[Feat.INCIDENCIA_100K],
        sub[Feat.DENSIDADE_KM2],
        s=120,
        label=str(year),
    )
ax.set_xlabel("Incidência média / 100 mil (por cluster)")
ax.set_ylabel("Densidade média (hab/km²)")
ax.set_yscale("log")
ax.set_title("Perfis v5 — centroides por ano")
ax.legend(title="ano", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Ato 4 — Transições v5 (semana t→t+1)

Compara dois anos: um com poucas notificações e o ano mais recente.

In [ ]:
low_year = int(coverage.loc[coverage["notificações"].idxmin(), "ano"])
high_year = int(coverage.loc[coverage["notificações"].idxmax(), "ano"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, year in zip(axes, [low_year, high_year]):
    r = run_year_version(year, VERSION_FINAL, keep_panel=True)
    trans = transition_matrix(r["panel"], r["labels"])
    sns.heatmap(trans, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"v5 — {year} ({coverage.set_index('ano').loc[year, 'notificações']:,} notif.)")
plt.tight_layout()

## Ato 5 — Escada completa no ano mais recente

Detalhe igual ao notebook de 2024, no último ano da série.

In [ ]:
last_year = YEARS[-1]
last_runs = [metrics_by[(last_year, v)] for v in VERSIONS]

evo_last = pd.DataFrame([{
    "versão": r["versão"],
    "bloco": r["bloco"],
    "silhouette": r["silhouette"],
    "n_feat": len(FEATURE_SETS[r["versão"]]),
} for r in last_runs])
evo_last["Δ silhouette"] = evo_last["silhouette"].diff().round(4)
display(evo_last.round(4))

run_final = run_year_version(last_year, VERSION_FINAL, keep_panel=True)
labeled = run_final["panel"].assign(cluster=run_final["labels"])
colors = labeled["cluster"].map({i: PALETTE[i] for i in range(K)})

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(
    labeled[Feat.INCIDENCIA_100K],
    labeled[Feat.DENSIDADE_KM2],
    c=colors,
    **SCATTER_KW,
)
ax.set_xlabel("Incidência / 100 mil")
ax.set_ylabel("Densidade (hab/km²)")
ax.set_yscale("log")
ax.set_title(f"v5 — nuvem município×semana ({last_year})")
plt.tight_layout()

## Conclusão

1. **10 anos** permitem ver se a escada v0→v5 é estável ou depende do volume epidêmico do ano.
2. Silhouette de v0 tende a ser alta em todos os anos (clusterização quase ordinal por casos).
3. v5 mantém leitura territorial (densidade) com pequena queda de métrica interna, como em 2024 isolado.
4. Transições e perfis mudam com a escala de notificações: comparar anos calmos vs anos de pico.